# Step 4: Tianmouc 仿真器

仿真器可以用普通 RGB 图像构造近似 AOP/COP 输入，帮助理解数据流。

本节直接使用：
- `run_sim_singleimg`
- `visualize_diff`

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / 'tianmoucv').exists() and (candidate / 'tianmoucv_example').exists():
        repo_root = candidate
        break
if repo_root is None:
    raise RuntimeError('找不到 tianmoucv 仓库根目录，请从仓库内打开本 Notebook。')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib.pyplot as plt
import torch


In [ ]:
from tianmoucv.data import TianmoucDataReader
from tianmoucv.sim import run_sim_singleimg, visualize_diff
from tianmoucv.utils import ensure_tutorial_dataset

info = ensure_tutorial_dataset()
dataset = TianmoucDataReader(str(info.sample_dir), N=1, print_info=False)
sample = dataset[0]
img0 = sample['F0'].numpy() * 255
img1 = sample['F1'].numpy() * 255

In [ ]:
img_pre, img_cur, td, sd0, sd1 = run_sim_singleimg(
    img_target=img0,
    img_ref=img1,
    sensor_width=img0.shape[1],
    sensor_height=img0.shape[0],
    xy=False,
    interp=True,
    device=torch.device('cpu'),
)

plt.figure(figsize=(14, 8))
plt.subplot(2, 3, 1)
plt.imshow(img0 / 255)
plt.title('Input F0')
plt.axis('off')

plt.subplot(2, 3, 2)
plt.imshow(img_pre.permute(1, 2, 0).numpy())
plt.title('Simulated Pre')
plt.axis('off')

plt.subplot(2, 3, 3)
plt.imshow(img_cur.permute(1, 2, 0).numpy())
plt.title('Simulated Cur')
plt.axis('off')

plt.subplot(2, 3, 4)
plt.imshow(visualize_diff(td.numpy(), vis_gain=4))
plt.title('TD(t=0~1)')
plt.axis('off')


plt.subplot(2, 3, 5)
plt.imshow(visualize_diff(sd0[0,...].numpy(), vis_gain=4))
plt.title('SD-SDL(t=0)')
plt.axis('off')

plt.subplot(2, 3, 6)
plt.imshow(visualize_diff(sd1[0,...].numpy(), vis_gain=4))
plt.title('SD-SDL(t=1)')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
img_pre_xy, img_cur_xy, td_xy, sdxy0, sdxy1 = run_sim_singleimg(
    img_target=img0,
    img_ref=img1,
    sensor_width=img0.shape[1],
    sensor_height=img0.shape[0],
    xy=True,
    interp=True,
    device=torch.device('cpu'),
)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(visualize_diff(sdxy0[0,...].numpy(), vis_gain=4))
plt.title('SDx(t=0)')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(visualize_diff(sdxy1[0,...].numpy(), vis_gain=4))
plt.title('SDx(t=1)')
plt.axis('off')
plt.tight_layout()
plt.show()